# 07 - Long-History Flood Panel & Administrative Taxonomy Drift

**Title**: `07_flood_history_and_taxonomy_drift.ipynb`  
**Purpose**: Audit panel integrity for the 16-year daily flood complaint series (2010 to 2026), perform cross-dataset reconciliation for Brooklyn, analyze multi-span precipitation coupling and predictive lead/lag relationships, evaluate time-split stationarity and extreme storm dominance, and quantify administrative taxonomy drift between legacy Sewer and Sewer Maintenance complaint types.  
**Inputs**: `data/processed/flood_daily_all_years.csv`, `data/processed/rain_daily.csv`, `data/processed/flood_raw_aggregate_2010_2019.csv`, `data/processed/flood_raw_aggregate_2020_onward.csv`, `data/processed/flood_descriptor_by_year.csv`, `data/processed/flood_type_discovery.csv`, `data/processed/complaints_clean.csv`, `data/processed/311_cube.csv`, `data/processed/report_tables/t26_3h_gap_summary.csv`, `data/processed/report_tables/t26_3h_missing_coords_descriptors.csv`  
**Outputs**: 16-year panel summary metrics, yearly pivot tables, boundary transition table, Brooklyn cross-check reconciliation, multi-span precipitation correlation profiles, lead/lag correlation spectrum, dry-spike storm anticipation table, 80/20 train/test partition storm metrics, city-wide monthly taxonomy drift table, descriptor transition matrix, baseline dry-spell comparison tables, 311 cube control ratios, diagnostic drift and dry-spell figures.  
**Runtime**: ~20-30 seconds

In [1]:
import sys; from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists()); sys.path.insert(0, str(ROOT))
from src.nbutils import Checker, md
RUN_HEAVY = False
RUN_API = False

In [2]:
import numpy as np
import pandas as pd
from src.preprocessing.flood_taxonomy import (
    load_flood_panel,
    build_sewer_backup_daily,
    check_panel_integrity,
    cross_check_brooklyn,
    compute_rain_relationships,
    compute_lead_tests,
    compute_storm_days_and_split,
    compute_taxonomy_drift,
    plot_taxonomy_drift,
    plot_dry_spell_comparison,
    BOROUGH_ORDER,
)

panel_path = ROOT / "data/processed/flood_daily_all_years.csv"
rain_path = ROOT / "data/processed/rain_daily.csv"
agg_2010_path = ROOT / "data/processed/flood_raw_aggregate_2010_2019.csv"
agg_2020_path = ROOT / "data/processed/flood_raw_aggregate_2020_onward.csv"
desc_path = ROOT / "data/processed/flood_descriptor_by_year.csv"
type_path = ROOT / "data/processed/flood_type_discovery.csv"
clean_path = ROOT / "data/processed/complaints_clean.csv"
cube_path = ROOT / "data/processed/311_cube.csv"
gap_summary_path = ROOT / "data/processed/report_tables/t26_3h_gap_summary.csv"
missing_desc_path = ROOT / "data/processed/report_tables/t26_3h_missing_coords_descriptors.csv"

print(f"Loading panel data from {panel_path.relative_to(ROOT)}...")
df_panel = load_flood_panel(panel_path)
print(f"Panel loaded: {len(df_panel):,} rows across {df_panel['day'].nunique():,} unique days.")

print("Building daily sewer_backup series from aggregate tables...")
df_sewer_backup = build_sewer_backup_daily(agg_2010_path, agg_2020_path)
print(f"Daily sewer_backup series constructed: {len(df_sewer_backup):,} rows, {df_sewer_backup['n'].sum():,} total complaints.")

Loading panel data from data\processed\flood_daily_all_years.csv...
Panel loaded: 30,740 rows across 6,148 unique days.
Building daily sewer_backup series from aggregate tables...


Daily sewer_backup series constructed: 30,740 rows, 218,439 total complaints.


In [3]:
print("=== Section 1: Panel Integrity & Historical Continuity ===")
integrity_res = check_panel_integrity(df_panel, desc_path, type_path)

print(f"Total Rows: {integrity_res['n_rows']:,}")
print(f"Unique Days: {integrity_res['n_days']:,}")
print(f"Duplicate (day, borough) pairs: {integrity_res['n_dupes']}")
print(f"Total Flood Complaints: {integrity_res['total_flood']:,}")

print("\n--- Flood Complaints per Borough per Year ---")
print(integrity_res["yearly_pivot"].to_string())

print("\n--- Ratio Flags (>3x or <1/3x of previous year) ---")
flags_df = integrity_res["ratio_flags"]
flagged_years = flags_df[flags_df.any(axis=1)]
if len(flagged_years) == 0:
    print("None: No year-over-year volume changes exceeded 3x or fell below 1/3x in any borough.")
else:
    print(flagged_years.to_string())

print("\n--- Descriptors First and Last Year Summary (Top 10) ---")
print(integrity_res["desc_summary"].head(10).to_string(index=False))

print("\n--- Complaint Types Discovered per Year (Top 15) ---")
print(integrity_res["df_types"].head(15).to_string(index=False))

print("\n--- Boundary Table: 2019-12-25 to 2020-01-05 ---")
print(integrity_res["boundary_table"].to_string())

=== Section 1: Panel Integrity & Historical Continuity ===
Total Rows: 30,740
Unique Days: 6,148
Duplicate (day, borough) pairs: 0
Total Flood Complaints: 200,259

--- Flood Complaints per Borough per Year ---
borough  BRONX  BROOKLYN  MANHATTAN  QUEENS  STATEN ISLAND
year                                                      
2010      1587      3107       1229    5265           1840
2011      1619      3642       1720    6135           2635
2012      1186      3099       1223    5263           1656
2013       969      2485       1135    3982           1633
2014      1166      3285       1476    4989           1935
2015       970      2509       1188    3878           1471
2016      1136      2365       1071    4438           1282
2017      1018      2702       1051    3878           1667
2018      1372      3586       1451    5110           2218
2019      1024      3114       1045    4577           1776
2020       798      2446        605    3188           1018
2021      1472      360

In [4]:
print("=== Section 2: Brooklyn Cross-Check (2025-11-01 to 2026-10-31) ===")
# Note: Provenance of t26_3h tables: Socrata aggregate queries run earlier.
crosscheck_res = cross_check_brooklyn(df_panel, clean_path, gap_summary_path, missing_desc_path)

print(f"Panel 12-Month Total: {crosscheck_res['panel_total']:,}")
print(f"Complaints Clean Total: {crosscheck_res['clean_total']:,}")
print(f"Days that Differ: {crosscheck_res['days_differ']}")
print(f"Largest Daily Difference: {crosscheck_res['max_daily_diff']}")
print(f"Pearson Correlation r: {crosscheck_res['pearson_r']:.6f}")

print("\n--- Top 10 Daily Discrepancies ---")
print(crosscheck_res["top_diff_days"].to_string(index=False))

print("\n--- Missing Coordinate Accounting (Provenance: Socrata aggregate queries run earlier) ---")
print(crosscheck_res["gap_summary"].to_string(index=False))

print("\n--- Missing Coordinates Descriptors Breakdown ---")
print(crosscheck_res["missing_desc"].to_string(index=False))

=== Section 2: Brooklyn Cross-Check (2025-11-01 to 2026-10-31) ===


Panel 12-Month Total: 5,614
Complaints Clean Total: 5,520
Days that Differ: 52
Largest Daily Difference: 10
Pearson Correlation r: 0.999586

--- Top 10 Daily Discrepancies ---
       day   n  clean_n  abs_diff
2026-07-06 238      228        10
2026-08-20  94       87         7
2026-05-20 149      145         4
2026-08-03 101       97         4
2026-09-27  48       44         4
2026-10-11  45       41         4
2026-10-16  15       11         4
2026-05-24 132      129         3
2026-08-23  79       76         3
2026-09-13  20       17         3

--- Missing Coordinate Accounting (Provenance: Socrata aggregate queries run earlier) ---
                                                          query_case  total_sewer_complaints  flood_group_complaints
Coordinates Present (latitude IS NOT NULL AND longitude IS NOT NULL)                   10832                    5520
         Coordinates Missing (latitude IS NULL OR longitude IS NULL)                     123                      94
        

In [5]:
print("=== Section 3: Rain Relationship Across Spans ===")
df_spans, df_bk_detailed = compute_rain_relationships(df_panel, rain_path)

print("\n--- Correlations Across Three Spans per Borough ---")
for span_name in ["full_span", "2020_onward", "last_12m"]:
    print(f"\nSpan: {span_name}")
    sub_span = df_spans[df_spans["span"] == span_name].drop(columns=["span"])
    print(sub_span.to_string(index=False))

print("\n--- Brooklyn Full Span Detailed Predictor Coupling ---")
print(df_bk_detailed.to_string(index=False))

=== Section 3: Rain Relationship Across Spans ===



--- Correlations Across Three Spans per Borough ---

Span: full_span
      borough  pearson_same_day  spearman_same_day  pearson_prev_day  spearman_prev_day  pearson_3d_sum  spearman_3d_sum
        BRONX          0.474100           0.263028          0.173649           0.147288        0.369899         0.262948
     BROOKLYN          0.442454           0.350697          0.137509           0.216327        0.330448         0.347116
    MANHATTAN          0.510925           0.317541          0.175569           0.204460        0.380824         0.301494
       QUEENS          0.534301           0.344467          0.194211           0.236651        0.412262         0.356454
STATEN ISLAND          0.490743           0.294746          0.211701           0.196999        0.413997         0.327747

Span: 2020_onward
      borough  pearson_same_day  spearman_same_day  pearson_prev_day  spearman_prev_day  pearson_3d_sum  spearman_3d_sum
        BRONX          0.453187           0.250182          0.18

In [6]:
print("=== Section 4: Lead/Lag Causality & Dry Spike Storm Anticipation ===")
df_leads, df_spikes = compute_lead_tests(df_panel, rain_path)

print("\n--- Lead/Lag Correlations (Day t flood vs Day t+k rain) ---")
lead_pivot_p = df_leads.pivot_table(index="offset_days", columns="borough", values="pearson_r")[BOROUGH_ORDER]
lead_pivot_s = df_leads.pivot_table(index="offset_days", columns="borough", values="spearman_r")[BOROUGH_ORDER]

print("Pearson r by Day Offset (-3 to +3):")
print(lead_pivot_p.to_string())
print("\nSpearman r by Day Offset (-3 to +3):")
print(lead_pivot_s.to_string())

print("\n--- Dry Spike Days Summary (flood >= p99 and rain < 5 mm) ---")
print(df_spikes.to_string(index=False))

=== Section 4: Lead/Lag Causality & Dry Spike Storm Anticipation ===



--- Lead/Lag Correlations (Day t flood vs Day t+k rain) ---
Pearson r by Day Offset (-3 to +3):
borough         BRONX  BROOKLYN  MANHATTAN    QUEENS  STATEN ISLAND
offset_days                                                        
-3           0.032755  0.030728   0.020470  0.038841       0.034268
-2           0.033750  0.030048   0.014990  0.039997       0.058388
-1           0.173649  0.137509   0.175569  0.194211       0.211701
 1           0.052771  0.058967   0.055326  0.067248       0.063229
 2          -0.001379  0.004722  -0.006244  0.002485       0.009642
 3           0.006290  0.015274   0.010502  0.018003       0.018607

Spearman r by Day Offset (-3 to +3):
borough         BRONX  BROOKLYN  MANHATTAN    QUEENS  STATEN ISLAND
offset_days                                                        
-3           0.048351  0.059297   0.025781  0.045130       0.047599
-2           0.061422  0.073596   0.038901  0.076750       0.093277
-1           0.147288  0.216327   0.204460  0.236

In [7]:
print("=== Section 5: Storm Days & Brooklyn Time Split Preview ===")
storm_res = compute_storm_days_and_split(df_panel, rain_path)

print(f"Storm Days (>=20 mm) Full Span: {dict(zip(BOROUGH_ORDER, storm_res['storm_full']))}")
print(f"Storm Days Last 12 Months: {dict(zip(BOROUGH_ORDER, storm_res['storm_12m']))}")
print(f"Storm Days Latest 20% Part: {dict(zip(BOROUGH_ORDER, storm_res['storm_test']))}")
print(f"Latest 20% Length: {storm_res['test_length']} days (starting {storm_res['test_start_date']})")
print(f"Earliest 80% Length: {storm_res['train_length']} days")

print(f"\nBrooklyn Storm Days: Train (80%) = {storm_res['bk_storm_train']}, Test (20%) = {storm_res['bk_storm_test']}")

print("\n--- Top 5 Flood-Count Days in Test Partition (Brooklyn) ---")
print(storm_res["top5_test"].to_string(index=False))

print(f"\nTotal Squared Deviation of Test Part: {storm_res['total_sq_dev']:,.2f}")
print(f"Top 2 Days Squared Deviation: {storm_res['top2_sq_dev']:,.2f}")
print(f"Share Contributed by Top 2 Days: {storm_res['share_top2_dev']:.4f}%")

=== Section 5: Storm Days & Brooklyn Time Split Preview ===


Storm Days (>=20 mm) Full Span: {'BRONX': 282, 'BROOKLYN': 282, 'MANHATTAN': 280, 'QUEENS': 269, 'STATEN ISLAND': 280}
Storm Days Last 12 Months: {'BRONX': 17, 'BROOKLYN': 19, 'MANHATTAN': 17, 'QUEENS': 18, 'STATEN ISLAND': 16}
Storm Days Latest 20% Part: {'BRONX': 71, 'BROOKLYN': 70, 'MANHATTAN': 70, 'QUEENS': 66, 'STATEN ISLAND': 65}
Latest 20% Length: 1230 days (starting 2023-06-20)
Earliest 80% Length: 4918 days

Brooklyn Storm Days: Train (80%) = 212, Test (20%) = 70

--- Top 5 Flood-Count Days in Test Partition (Brooklyn) ---
       day   n  precip_mm
2023-09-29 970       62.1
2025-10-30 584       40.0
2026-07-06 238       49.6
2026-05-21 150        7.2
2026-05-20 149        7.2

Total Squared Deviation of Test Part: 1,528,458.99
Top 2 Days Squared Deviation: 1,244,955.46
Share Contributed by Top 2 Days: 81.4517%


In [8]:
print("=== Section 6: Municipal Taxonomy Drift & Dry-Spell Benchmarking ===")
drift_res = compute_taxonomy_drift(agg_2020_path, rain_path, df_panel, df_sewer_backup, cube_path)

print("\n--- City-Wide Monthly Taxonomy Drift (2025-01 to 2026-10) ---")
print(drift_res["df_drift"].to_string(index=False))

print("\n--- Flood Descriptors by Month (2026-04 to 2026-10) ---")
print(drift_res["df_desc_monthly"].to_string(index=False))

print("\n--- Dry-Day Baseline (precip_mm < 2) per Borough ---")
print(drift_res["df_baseline"].to_string(index=False))

print("\n--- Cleaner Dry-Spell Test per Borough (Aug to Oct, 2023-2026) ---")
print(drift_res["df_dry_spell"].to_string(index=False))

print("\n--- 311 Cube Municipal Volume Control Ratios ---")
print(drift_res["controls_summary"].to_string(index=False))

=== Section 6: Municipal Taxonomy Drift & Dry-Spell Benchmarking ===



--- City-Wide Monthly Taxonomy Drift (2025-01 to 2026-10) ---
  month  n_sewer  n_sewer_maintenance  n_flood  n_sewer_backup  flood_share  city_rain_sum
2025-01     1424                    1      412             617     0.400389          26.50
2025-02     1603                    1      473             706     0.401187          79.60
2025-03     2014                    3      712             751     0.486671         137.92
2025-04     1741                    0      628             579     0.520298          79.68
2025-05     2498                    5     1203             687     0.636508         195.24
2025-06     2014                    9      824             590     0.582744          71.38
2025-07     2739                    6     1204             811     0.597519         118.32
2025-08     2475                    4     1025             776     0.569128          24.18
2025-09     1849                   12      626             525     0.543875         101.44
2025-10     3548           

In [9]:
fig1_path = ROOT / "report/figures/07_monthly_taxonomy_drift.png"
plot_taxonomy_drift(drift_res["df_drift"], fig1_path)
print(f"Saved figure: {fig1_path.relative_to(ROOT)}")

fig2_path = ROOT / "report/figures/07_dry_spell_comparison.png"
plot_dry_spell_comparison(drift_res["df_dry_spell"], fig2_path)
print(f"Saved figure: {fig2_path.relative_to(ROOT)}")

Saved figure: report\figures\07_monthly_taxonomy_drift.png


Saved figure: report\figures\07_dry_spell_comparison.png


In [10]:
checker = Checker()

# 1. panel rows 30740, days 6148, duplicate (day, borough) pairs 0, total flood n 200259
checker.check("panel rows = 30740", len(df_panel), 30740)
checker.check("days = 6148", int(df_panel["day"].nunique()), 6148)
checker.check("duplicate (day, borough) pairs = 0", int(df_panel.duplicated(subset=["day", "borough"]).sum()), 0)
checker.check("total flood n = 200259", int(df_panel["n"].sum()), 200259)

# 2. Brooklyn flood per year 2010=3107, 2018=3586, 2023=4399, 2025=3500, 2026=4963
bk_panel = df_panel[df_panel["borough"] == "BROOKLYN"].copy()
bk_panel["year"] = bk_panel["day"].str[:4].astype(int)
bk_yearly = bk_panel.groupby("year")["n"].sum()
bk_years_actual = [int(bk_yearly.get(y, 0)) for y in [2010, 2018, 2023, 2025, 2026]]
checker.check("Brooklyn flood per year [3107, 3586, 4399, 3500, 4963]", bk_years_actual, [3107, 3586, 4399, 3500, 4963])

# 3. Brooklyn 2019-12-30 = 9 and 2020-01-04 = 7
bk_d1 = int(bk_panel[bk_panel["day"] == "2019-12-30"]["n"].values[0])
bk_d2 = int(bk_panel[bk_panel["day"] == "2020-01-04"]["n"].values[0])
checker.check("Brooklyn 2019-12-30 = 9", bk_d1, 9)
checker.check("Brooklyn 2020-01-04 = 7", bk_d2, 7)

# 4. Brooklyn window panel total 5614, complaints_clean total 5520, days that differ 52, largest daily difference 10, Pearson 0.999586
checker.check("Brooklyn window panel total = 5614", int(crosscheck_res["panel_total"]), 5614)
checker.check("complaints_clean total = 5520", int(crosscheck_res["clean_total"]), 5520)
checker.check("days that differ = 52", int(crosscheck_res["days_differ"]), 52)
checker.check("largest daily difference = 10", int(crosscheck_res["max_daily_diff"]), 10)
checker.check("cross-check Pearson r = 0.999586", round(float(crosscheck_res["pearson_r"]), 6), 0.999586, tol=0.0005)

# 5. Brooklyn full-span Pearson same-day 0.4425, Spearman same-day 0.3507, Pearson 3-day 0.3304, Spearman 3-day 0.3471, Pearson with precip_hours 0.3281, log1p Pearson with precip_mm 0.4364 and with the 3-day sum 0.4023
# Queens full-span Pearson same-day 0.5343 and Spearman same-day 0.3445
bk_spans = df_spans[(df_spans["span"] == "full_span") & (df_spans["borough"] == "BROOKLYN")].iloc[0]
qn_spans = df_spans[(df_spans["span"] == "full_span") & (df_spans["borough"] == "QUEENS")].iloc[0]

checker.check("Brooklyn full-span Pearson same-day = 0.4425", round(float(bk_spans["pearson_same_day"]), 4), 0.4425, tol=0.0005)
checker.check("Brooklyn full-span Spearman same-day = 0.3507", round(float(bk_spans["spearman_same_day"]), 4), 0.3507, tol=0.0005)
checker.check("Brooklyn full-span Pearson 3-day = 0.3304", round(float(bk_spans["pearson_3d_sum"]), 4), 0.3304, tol=0.0005)
checker.check("Brooklyn full-span Spearman 3-day = 0.3471", round(float(bk_spans["spearman_3d_sum"]), 4), 0.3471, tol=0.0005)

row_hours = df_bk_detailed[df_bk_detailed["predictor"] == "precip_hours"].iloc[0]
row_mm = df_bk_detailed[df_bk_detailed["predictor"] == "precip_mm"].iloc[0]
row_3d_mm = df_bk_detailed[df_bk_detailed["predictor"] == "3-day sum precip_mm"].iloc[0]

checker.check("Brooklyn Pearson with precip_hours = 0.3281", round(float(row_hours["flood_count_pearson"]), 4), 0.3281, tol=0.0005)
checker.check("Brooklyn log1p Pearson with precip_mm = 0.4364", round(float(row_mm["log1p_flood_count_pearson"]), 4), 0.4364, tol=0.0005)
checker.check("Brooklyn log1p Pearson with 3-day sum = 0.4023", round(float(row_3d_mm["log1p_flood_count_pearson"]), 4), 0.4023, tol=0.0005)

checker.check("Queens full-span Pearson same-day = 0.5343", round(float(qn_spans["pearson_same_day"]), 4), 0.5343, tol=0.0005)
checker.check("Queens full-span Spearman same-day = 0.3445", round(float(qn_spans["spearman_same_day"]), 4), 0.3445, tol=0.0005)

# 6. storm days full span [282,282,280,269,280], last 12 months [17,19,17,18,16], latest 20% [71,70,70,66,65], the latest-20% length 1230 days starting 2023-06-20
checker.check("storm days full span [282,282,280,269,280]", storm_res["storm_full"], [282, 282, 280, 269, 280])
checker.check("storm days last 12m [17,19,17,18,16]", storm_res["storm_12m"], [17, 19, 17, 18, 16])
checker.check("storm days latest 20% [71,70,70,66,65]", storm_res["storm_test"], [71, 70, 70, 66, 65])
checker.check("latest-20% length = 1230 days", int(storm_res["test_length"]), 1230)
checker.check("latest-20% start date = 2023-06-20", str(storm_res["test_start_date"]), "2023-06-20")

# 7. city aggregates: Sewer 2025-08 = 2475, Sewer Maintenance 2025-08 = 4, Sewer 2026-07 = 2983, Sewer Maintenance 2026-07 = 1455, Sewer 2026-08 = 0, Sewer Maintenance 2026-08 = 5060, flood n 2026-08 = 2622, sewer_backup n 2026-08 = 1540, flood share 2026-08 = 0.629986
df_drift_indexed = drift_res["df_drift"].set_index("month")
checker.check("Sewer 2025-08 = 2475", int(df_drift_indexed.loc["2025-08", "n_sewer"]), 2475)
checker.check("Sewer Maintenance 2025-08 = 4", int(df_drift_indexed.loc["2025-08", "n_sewer_maintenance"]), 4)
checker.check("Sewer 2026-07 = 2983", int(df_drift_indexed.loc["2026-07", "n_sewer"]), 2983)
checker.check("Sewer Maintenance 2026-07 = 1455", int(df_drift_indexed.loc["2026-07", "n_sewer_maintenance"]), 1455)
checker.check("Sewer 2026-08 = 0", int(df_drift_indexed.loc["2026-08", "n_sewer"]), 0)
checker.check("Sewer Maintenance 2026-08 = 5060", int(df_drift_indexed.loc["2026-08", "n_sewer_maintenance"]), 5060)
checker.check("flood n 2026-08 = 2622", int(df_drift_indexed.loc["2026-08", "n_flood"]), 2622)
checker.check("sewer_backup n 2026-08 = 1540", int(df_drift_indexed.loc["2026-08", "n_sewer_backup"]), 1540)
checker.check("flood share 2026-08 = 0.629986", round(float(df_drift_indexed.loc["2026-08", "flood_share"]), 6), 0.629986, tol=0.0005)

# 8. Brooklyn dry-day baseline A days 407, A mean 7.6855, B days 62, B mean 13.0484, ratio 1.6978 (tol 0.001)
bk_base = drift_res["df_baseline"][drift_res["df_baseline"]["borough"] == "BROOKLYN"].iloc[0]
checker.check("Brooklyn dry baseline A days = 407", int(bk_base["A_days"]), 407)
checker.check("Brooklyn dry baseline A mean = 7.6855", round(float(bk_base["A_mean"]), 4), 7.6855, tol=0.0005)
checker.check("Brooklyn dry baseline B days = 62", int(bk_base["B_days"]), 62)
checker.check("Brooklyn dry baseline B mean = 13.0484", round(float(bk_base["B_mean"]), 4), 13.0484, tol=0.0005)
checker.check("Brooklyn dry baseline ratio = 1.6978", round(float(bk_base["B_over_A_ratio"]), 4), 1.6978, tol=0.001)

checker.summary()

[PASS] panel rows = 30740: actual=30740, expected=30740
[PASS] days = 6148: actual=6148, expected=6148
[PASS] duplicate (day, borough) pairs = 0: actual=0, expected=0
[PASS] total flood n = 200259: actual=200259, expected=200259
[PASS] Brooklyn flood per year [3107, 3586, 4399, 3500, 4963]: actual=[3107, 3586, 4399, 3500, 4963], expected=[3107, 3586, 4399, 3500, 4963]
[PASS] Brooklyn 2019-12-30 = 9: actual=9, expected=9
[PASS] Brooklyn 2020-01-04 = 7: actual=7, expected=7
[PASS] Brooklyn window panel total = 5614: actual=5614, expected=5614
[PASS] complaints_clean total = 5520: actual=5520, expected=5520
[PASS] days that differ = 52: actual=52, expected=52
[PASS] largest daily difference = 10: actual=10, expected=10
[PASS] cross-check Pearson r = 0.999586: actual=0.999586, expected=0.999586
[PASS] Brooklyn full-span Pearson same-day = 0.4425: actual=0.4425, expected=0.4425
[PASS] Brooklyn full-span Spearman same-day = 0.3507: actual=0.3507, expected=0.3507
[PASS] Brooklyn full-span Pea

In [11]:
findings_07 = f"""### Findings
- **16-Year Panel Integrity**: The long-history daily panel spans exactly {integrity_res['n_days']:,} calendar days ({len(df_panel):,} borough-day rows) from 2010-01-01 to 2026-10-31 with zero duplicate pairs and a cumulative city-wide volume of {integrity_res['total_flood']:,} flood incidents.
- **Year-Over-Year Volume Stability**: Annual flood complaint totals across all five boroughs display stable longitudinal dynamics, with no single borough-year transition exceeding the 3x or 1/3x drift threshold.
- **Brooklyn Clean Dataset Reconciliation**: Across the 365-day target extraction window, the daily panel aggregates to {crosscheck_res['panel_total']:,} complaints compared to {crosscheck_res['clean_total']:,} in `complaints_clean.csv`. Exactly {crosscheck_res['days_differ']} dates differ by at most {crosscheck_res['max_daily_diff']} incidents, yielding a near-perfect daily correlation of r = {crosscheck_res['pearson_r']:.6f}.
- **Unmapped Coordinate Accounting**: Query reconciliation confirms that all 94 missing complaints in the clean extract are accounted for by records lacking valid geographic coordinates in the municipal database.
- **Precipitation Coupling Spectrum**: Across the full 16-year timeline, daily flood volumes demonstrate substantial positive coupling with same-day precipitation, led by Queens (Pearson r = {qn_spans['pearson_same_day']:.4f}, Spearman r = {qn_spans['spearman_same_day']:.4f}) and Brooklyn (Pearson r = {bk_spans['pearson_same_day']:.4f}, Spearman r = {bk_spans['spearman_same_day']:.4f}).
- **Lead/Lag Asymmetry**: Predictive lead tests demonstrate that future precipitation (days t+1, t+2, t+3) displays markedly lower correlation with day t flood reports than antecedent or contemporaneous rainfall, confirming that reporting surges are responsive rather than anticipatory.
- **Dry Spike Storm Sensitivity**: Among rare dry-day volume spikes (flood count >= 99th percentile and precipitation < 5 mm), subsequent 3-day precipitation (>=20 mm) occurs in {df_spikes[df_spikes['borough'] == 'BROOKLYN']['share_next_3d_ge_20mm_pct'].values[0]:.2f}% of cases in Brooklyn relative to a {df_spikes[df_spikes['borough'] == 'BROOKLYN']['base_rate_all_days_ge_20mm_pct'].values[0]:.2f}% base rate.
- **Extreme Event Variance Dominance**: In the chronological 80/20 temporal split (test partition commencing {storm_res['test_start_date']}), the two highest flood-volume dates account for {storm_res['share_top2_dev']:.4f}% of the test partition's entire squared deviation from its mean.
- **Administrative Taxonomy Shift**: An administrative transition occurred in August 2026, wherein legacy 'Sewer' complaint intake transitioned completely to 'Sewer Maintenance' ({int(df_drift_indexed.loc['2026-08', 'n_sewer_maintenance']):,} complaints vs {int(df_drift_indexed.loc['2026-08', 'n_sewer']):,} under 'Sewer').
- **Dry-Spell Baseline Shift**: In Brooklyn, mean daily dry-day flood reports increased from {bk_base['A_mean']:.4f} in Period A (2025-01-01 to 2026-07-31) to {bk_base['B_mean']:.4f} in Period B (2026-08-01 to 2026-10-31), representing a ratio of {bk_base['B_over_A_ratio']:.4f}.
"""
md(findings_07)
checker.summary()

### Findings
- **16-Year Panel Integrity**: The long-history daily panel spans exactly 6,148 calendar days (30,740 borough-day rows) from 2010-01-01 to 2026-10-31 with zero duplicate pairs and a cumulative city-wide volume of 200,259 flood incidents.
- **Year-Over-Year Volume Stability**: Annual flood complaint totals across all five boroughs display stable longitudinal dynamics, with no single borough-year transition exceeding the 3x or 1/3x drift threshold.
- **Brooklyn Clean Dataset Reconciliation**: Across the 365-day target extraction window, the daily panel aggregates to 5,614 complaints compared to 5,520 in `complaints_clean.csv`. Exactly 52 dates differ by at most 10 incidents, yielding a near-perfect daily correlation of r = 0.999586.
- **Unmapped Coordinate Accounting**: Query reconciliation confirms that all 94 missing complaints in the clean extract are accounted for by records lacking valid geographic coordinates in the municipal database.
- **Precipitation Coupling Spectrum**: Across the full 16-year timeline, daily flood volumes demonstrate substantial positive coupling with same-day precipitation, led by Queens (Pearson r = 0.5343, Spearman r = 0.3445) and Brooklyn (Pearson r = 0.4425, Spearman r = 0.3507).
- **Lead/Lag Asymmetry**: Predictive lead tests demonstrate that future precipitation (days t+1, t+2, t+3) displays markedly lower correlation with day t flood reports than antecedent or contemporaneous rainfall, confirming that reporting surges are responsive rather than anticipatory.
- **Dry Spike Storm Sensitivity**: Among rare dry-day volume spikes (flood count >= 99th percentile and precipitation < 5 mm), subsequent 3-day precipitation (>=20 mm) occurs in 50.00% of cases in Brooklyn relative to a 17.79% base rate.
- **Extreme Event Variance Dominance**: In the chronological 80/20 temporal split (test partition commencing 2023-06-20), the two highest flood-volume dates account for 81.4517% of the test partition's entire squared deviation from its mean.
- **Administrative Taxonomy Shift**: An administrative transition occurred in August 2026, wherein legacy 'Sewer' complaint intake transitioned completely to 'Sewer Maintenance' (5,060 complaints vs 0 under 'Sewer').
- **Dry-Spell Baseline Shift**: In Brooklyn, mean daily dry-day flood reports increased from 7.6855 in Period A (2025-01-01 to 2026-07-31) to 13.0484 in Period B (2026-08-01 to 2026-10-31), representing a ratio of 1.6978.


CHECKS: 40 passed, 0 failed


### Limitations

- Long-history municipal records prior to 2020 originate from an archived Socrata endpoint (`76ig-c548`), whereas records from 2020 onward originate from the live portal (`erm2-nwe9`), introducing potential discrepancies in administrative resolution workflows and classification metadata.
- Daily precipitation metrics are queried from borough centroid coordinates via the Open-Meteo historical archive, which aggregates rainfall over calendar days (midnight to midnight) rather than aligning with localized convective cloudburst timing or sub-daily drainage catchment saturation cycles.
- The administrative taxonomy transition between 'Sewer' and 'Sewer Maintenance' in August 2026 alters baseline intake rates, indicating that raw unadjusted complaint frequencies across this boundary should be interpreted in the context of institutional reporting changes rather than purely physical hydrological variation.